In [ ]:
!nvidia-smi

Sun Nov 15 17:21:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

PROJECT_DIR = "/content/stage4_dock_and_relax"

# Create required standard directories
folders = [
    f"{PROJECT_DIR}/configs",
    f"{PROJECT_DIR}/data/targets",
    f"{PROJECT_DIR}/data/receptors",
    f"{PROJECT_DIR}/data/ligands",
    f"{PROJECT_DIR}/scripts",
    f"{PROJECT_DIR}/outputs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"📁 Created: {folder}")

%cd {PROJECT_DIR}

📁 Created: /content/stage4_dock_and_relax/configs
📁 Created: /content/stage4_dock_and_relax/data/targets
📁 Created: /content/stage4_dock_and_relax/data/receptors
📁 Created: /content/stage4_dock_and_relax/data/ligands
📁 Created: /content/stage4_dock_and_relax/scripts
📁 Created: /content/stage4_dock_and_relax/outputs
/content/stage4_dock_and_relax


In [ ]:
# 1. Install Linux C++ build tools required by Vina
!apt-get update -qq && apt-get install -y -qq libboost-all-dev swig

# 2. Install Python biophysics & cheminformatics libraries
!pip install -q openmm rdkit meeko gemmi pandas numpy matplotlib scipy vina

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package javascript-common.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../000-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libboost1.83-tools-dev.
Preparing to unpack .../001-libboost1.83-tools-dev_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost1.83-tools-dev (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libboost-tools-dev.
Preparing to unpack .../002-libboost-tools-dev_1.83.0.1ubuntu2_amd64.deb ...
Unpacking libboost-tools-dev (1.83.0.1ubuntu2) ...
Selecting previously unselected package libboost-atomic1.83.0:amd64.
Preparing to unpack .../003-libboost-atomic1.83.0_1.83.0-2.1ubunt

In [ ]:
!pip install -q pdbfixer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import openmm as mm

platforms = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print("✅ Available OpenMM Platforms:", platforms)
assert any(p in platforms for p in ['CUDA', 'OpenCL']), "Warning: GPU platform not detected; CPU fallback will be slow."

✅ Available OpenMM Platforms: ['Reference', 'CPU', 'OpenCL']


In [ ]:
%%writefile scripts/local_dock.py
import argparse
import os
import numpy as np
import pandas as pd
from vina import Vina
from rdkit import Chem
from rdkit.Chem import AllChem, rdDetermineBonds
from meeko import MoleculePreparation, PDBQTMolecule, RDKitMolCreate

def prepare_ligand_to_pdbqt(input_path, out_dir):
    """Auto-detects .npz, .sdf, or .pdbqt and prepares a valid Vina PDBQT."""
    if input_path.endswith('.pdbqt'):
        return input_path, None

    mol = None
    if input_path.endswith('.npz'):
        data = np.load(input_path, allow_pickle=True)
        coords = data['pos'] if 'pos' in data else data['coords']
        atomic_nums = data['atom_types'] if 'atom_types' in data else data['elements']

        rwmol = Chem.RWMol()
        conf = Chem.Conformer(len(atomic_nums))
        for i, z in enumerate(atomic_nums):
            a_idx = rwmol.AddAtom(Chem.Atom(int(z)))
            conf.SetAtomPosition(a_idx, coords[i].tolist())
        rwmol.AddConformer(conf)
        mol = rwmol.GetMol()
        try:
            rdDetermineBonds.DetermineConnectivity(mol)
            rdDetermineBonds.DetermineBondOrders(mol, charge=0)
        except Exception:
            pass
    elif input_path.endswith('.sdf'):
        suppl = Chem.SDMolSupplier(input_path, removeHs=False)
        mol = suppl[0] if suppl else None

    if mol is None:
        raise ValueError(f"Could not parse molecule from {input_path}")

    # Prepare to PDBQT using standard Meeko API
    mol = Chem.AddHs(mol, addCoords=True)
    prep = MoleculePreparation()
    prep.prepare(mol)
    pdbqt_str = prep.write_pdbqt_string()

    pdbqt_out = os.path.join(out_dir, "prepared_ligand.pdbqt")
    with open(pdbqt_out, "w") as f:
        f.write(pdbqt_str)
    return pdbqt_out, mol

def run_local_docking(receptor_pdbqt, ligand_path, center, size, outdir, exhaustiveness=8, num_modes=9):
    os.makedirs(outdir, exist_ok=True)

    # 1. Prepare ligand
    ligand_pdbqt, original_mol = prepare_ligand_to_pdbqt(ligand_path, outdir)

    # 2. Configure Vina
    v = Vina(sf_name='vina')
    v.set_receptor(receptor_pdbqt)
    v.set_ligand_from_file(ligand_pdbqt)

    # 3. Compute grid maps & local search
    v.compute_vina_maps(center=center, box_size=size)
    v.dock(exhaustiveness=exhaustiveness, n_poses=num_modes)

    # 4. Save poses and scores
    docked_pdbqt_path = os.path.join(outdir, "docked_poses.pdbqt")
    v.write_poses(docked_pdbqt_path, n_poses=num_modes, overwrite=True)

    energies = v.energies(n_poses=num_modes)
    scores = [{"mode": i + 1, "affinity_kcal_mol": e[0], "rmsd_lb": e[1], "rmsd_ub": e[2]} for i, e in enumerate(energies)]
    pd.DataFrame(scores).to_csv(os.path.join(outdir, "docking_scores.csv"), index=False)

    # 5. Export Mode 1 to docked_pose.sdf
    docked_sdf_path = os.path.join(outdir, "docked_pose.sdf")
    try:
        pdbqt_mol = PDBQTMolecule.from_file(docked_pdbqt_path, is_dlg=False)
        mols = RDKitMolCreate.from_pdbqt_mol(pdbqt_mol)
        if mols and len(mols) > 0:
            w = Chem.SDWriter(docked_sdf_path)
            w.write(mols[0])
            w.close()
    except Exception:
        pass

    # Fail-safe Fallback: Transfer Vina Mode 1 coordinates onto original mol
    if not os.path.exists(docked_sdf_path) and original_mol is not None:
        coords = []
        with open(docked_pdbqt_path, "r") as f:
            for line in f:
                if line.startswith("MODEL 2") or line.startswith("ENDMDL"):
                    break
                if line.startswith("ATOM") or line.startswith("HETATM"):
                    coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        if len(coords) == original_mol.GetNumAtoms():
            conf = original_mol.GetConformer()
            for i, (x, y, z) in enumerate(coords):
                conf.SetAtomPosition(i, Chem.rdGeometry.Point3D(x, y, z))
            w = Chem.SDWriter(docked_sdf_path)
            w.write(original_mol)
            w.close()

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--receptor-pdbqt", required=True)
    parser.add_argument("--ligand-path", required=True)
    parser.add_argument("--center", nargs=3, type=float, required=True)
    parser.add_argument("--size", nargs=3, type=float, required=True)
    parser.add_argument("--outdir", required=True)
    parser.add_argument("--exhaustiveness", type=int, default=8)
    parser.add_argument("--num-modes", type=int, default=9)
    args = parser.parse_args()

    run_local_docking(args.receptor_pdbqt, args.ligand_path, args.center, args.size,
                      args.outdir, args.exhaustiveness, args.num_modes)

Overwriting scripts/local_dock.py


In [ ]:
%%writefile scripts/build_complex_system.py
import argparse
import json
import os
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import openmm as mm
from openmm import app, unit

def build_complex_system(target_clean_pdb, docked_sdf, outdir):
    os.makedirs(outdir, exist_ok=True)

    # 1. Load protein receptor structure
    prot_pdb = app.PDBFile(target_clean_pdb)
    forcefield = app.ForceField('amber14-all.xml')

    # 2. Create base OpenMM system for the protein
    system = forcefield.createSystem(
        prot_pdb.topology,
        nonbondedMethod=app.NoCutoff,
        constraints=app.HBonds
    )

    # 3. Load docked ligand from SDF
    suppl = Chem.SDMolSupplier(docked_sdf, removeHs=False)
    mol = suppl[0] if suppl else None
    if mol is None:
        raise ValueError(f"Cannot load docked SDF: {docked_sdf}")

    mol = Chem.AddHs(mol, addCoords=True)
    AllChem.ComputeGasteigerCharges(mol)
    conf = mol.GetConformer()

    n_prot_atoms = prot_pdb.topology.getNumAtoms()
    n_lig_atoms = mol.GetNumAtoms()

    protein_indices = list(range(n_prot_atoms))
    ligand_indices = list(range(n_prot_atoms, n_prot_atoms + n_lig_atoms))

    # 4. Add ligand particles to the OpenMM System
    nonbonded_force = None
    for f_idx in range(system.getNumForces()):
        f = system.getForce(f_idx)
        if isinstance(f, mm.NonbondedForce):
            nonbonded_force = f
            break

    vdw_radii = {'H': 0.12, 'C': 0.17, 'N': 0.155, 'O': 0.152, 'F': 0.147, 'P': 0.18, 'S': 0.18, 'Cl': 0.175, 'Br': 0.185, 'I': 0.198}

    for atom in mol.GetAtoms():
        mass = atom.GetMass()
        system.addParticle(mass * unit.amu)

        # Nonbonded parameters (charge, sigma, epsilon)
        elem = atom.GetSymbol()
        sigma = 2.0 * vdw_radii.get(elem, 0.17) * unit.nanometers
        epsilon = 0.40 * unit.kilojoules_per_mole
        try:
            q = float(atom.GetProp('_GasteigerCharge'))
            if np.isnan(q): q = 0.0
        except Exception:
            q = 0.0

        if nonbonded_force:
            nonbonded_force.addParticle(q * unit.elementary_charge, sigma, epsilon)

    # 5. Add ligand covalent bond constraints to maintain molecular topology
    bond_force = mm.HarmonicBondForce()
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx() + n_prot_atoms
        j = bond.GetEndAtomIdx() + n_prot_atoms
        p1 = conf.GetAtomPosition(bond.GetBeginAtomIdx())
        p2 = conf.GetAtomPosition(bond.GetEndAtomIdx())
        dist = np.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2 + (p1.z - p2.z)**2) * 0.1 # Angstrom to nm
        bond_force.addBond(i, j, dist * unit.nanometer, 250000.0 * unit.kilojoules_per_mole / (unit.nanometer**2))
    system.addForce(bond_force)

    # 6. Save indices JSON files
    lig_idx_file = os.path.join(outdir, "ligand_indices.json")
    prot_idx_file = os.path.join(outdir, "protein_indices.json")
    with open(lig_idx_file, "w") as f: json.dump(ligand_indices, f)
    with open(prot_idx_file, "w") as f: json.dump(protein_indices, f)

    # 7. Write combined complex PDB file
    complex_pdb_file = os.path.join(outdir, "initial_complex.pdb")
    with open(target_clean_pdb, "r") as rf, open(complex_pdb_file, "w") as wf:
        for line in rf:
            if line.startswith("ATOM"):
                wf.write(line)
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            wf.write(f"HETATM{atom.GetIdx()+1:>5} {atom.GetSymbol():>2}   LIG L   1    "
                     f"{pos.x:8.3f}{pos.y:8.3f}{pos.z:8.3f}  1.00  0.00          {atom.GetSymbol():>2}\n")
        wf.write("END\n")

    # 8. Save serialized system XML
    system_xml_file = os.path.join(outdir, "system.xml")
    with open(system_xml_file, "w") as f:
        f.write(mm.XmlSerializer.serialize(system))

    print(f"  [System Builder] Protein atoms: {n_prot_atoms} | Ligand atoms: {n_lig_atoms}")
    return complex_pdb_file, system_xml_file, lig_idx_file, prot_idx_file

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--target-clean-pdb", required=True)
    parser.add_argument("--docked-sdf", required=True)
    parser.add_argument("--outdir", required=True)
    args = parser.parse_args()
    build_complex_system(args.target_clean_pdb, args.docked_sdf, args.outdir)

Overwriting scripts/build_complex_system.py


In [ ]:
%%writefile scripts/relax_complex.py
import argparse
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openmm as mm
from openmm import app, unit

def compute_displacement_rmsd(curr_coords, init_coords, ligand_indices):
    diff = curr_coords[ligand_indices] - init_coords[ligand_indices]
    return float(np.sqrt(np.mean(np.sum(diff**2, axis=-1))))

def run_relaxation(complex_pdb, system_xml, ligand_idx_path, protein_idx_path, outdir,
                   max_iterations=500, record_every=10, restraint_k=1000.0):
    os.makedirs(outdir, exist_ok=True)
    log_file = open(os.path.join(outdir, "run.log"), "w")

    def log(msg):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("="*60)
    log(f"Stage 4 OpenMM Relaxation: {complex_pdb}")
    log("="*60)

    # 1. Load files & perform validation checks
    with open(ligand_idx_path) as f: ligand_indices = json.load(f)
    with open(protein_idx_path) as f: protein_indices = json.load(f)
    with open(system_xml, 'r') as f: system = mm.XmlSerializer.deserialize(f.read())
    pdb = app.PDBFile(complex_pdb)

    n_pdb_atoms = pdb.topology.getNumAtoms()
    n_system_particles = system.getNumParticles()
    assert n_pdb_atoms == n_system_particles, f"Atom mismatch: PDB={n_pdb_atoms}, System={n_system_particles}"
    assert len(set(ligand_indices).intersection(set(protein_indices))) == 0, "Indices must be disjoint!"

    # 2. Apply Harmonic Restraints to Protein atoms
    restraint = mm.CustomExternalForce("0.5 * k * ((x-x0)^2 + (y-y0)^2 + (z-z0)^2)")
    restraint.addGlobalParameter("k", restraint_k * unit.kilojoule_per_mole / (unit.nanometer**2))
    restraint.addPerParticleParameter("x0")
    restraint.addPerParticleParameter("y0")
    restraint.addPerParticleParameter("z0")

    init_positions = pdb.positions
    for idx in protein_indices:
        pos = init_positions[idx].value_in_unit(unit.nanometers)
        restraint.addParticle(idx, [pos[0], pos[1], pos[2]])
    system.addForce(restraint)

    # 3. Initialize Context on GPU platform
    available = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
    platform_name = 'CUDA' if 'CUDA' in available else ('OpenCL' if 'OpenCL' in available else 'CPU')
    log(f"[PLATFORM] Using Platform: {platform_name}")
    platform = mm.Platform.getPlatformByName(platform_name)

    context = mm.Context(system, mm.VerletIntegrator(1.0 * unit.femtoseconds), platform)
    context.setPositions(init_positions)

    # 4. Record Step 0
    init_pos_arr = np.array(init_positions.value_in_unit(unit.angstroms))
    state0 = context.getState(getEnergy=True, getForces=True, getPositions=True)
    init_energy = state0.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
    forces0 = np.array(state0.getForces().value_in_unit(unit.kilojoule_per_mole / unit.nanometer))
    init_max_force = float(np.max(np.linalg.norm(forces0, axis=-1)))

    trace = [{
        "iteration": 0,
        "ligand_rmsd_A": 0.0,
        "potential_energy_kJ_mol": init_energy,
        "max_force_kJ_mol_nm": init_max_force
    }]

    # 5. Minimization Loop (Up to 500 iterations)
    total_steps = 0
    while total_steps < max_iterations:
        steps_to_take = min(record_every, max_iterations - total_steps)
        mm.LocalEnergyMinimizer.minimize(context, maxIterations=steps_to_take)
        total_steps += steps_to_take

        state = context.getState(getEnergy=True, getForces=True, getPositions=True)
        curr_pos = np.array(state.getPositions().value_in_unit(unit.angstroms))
        energy = state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
        forces = np.array(state.getForces().value_in_unit(unit.kilojoule_per_mole / unit.nanometer))
        max_f = float(np.max(np.linalg.norm(forces, axis=-1)))
        rmsd = compute_displacement_rmsd(curr_pos, init_pos_arr, ligand_indices)

        trace.append({
            "iteration": total_steps,
            "ligand_rmsd_A": rmsd,
            "potential_energy_kJ_mol": energy,
            "max_force_kJ_mol_nm": max_f
        })

    # 6. Save QC Outputs
    df = pd.DataFrame(trace)
    df.to_csv(os.path.join(outdir, "relaxation_trace.csv"), index=False)

    relaxed_pdb = os.path.join(outdir, "relaxed_complex.pdb")
    with open(relaxed_pdb, "w") as f:
        app.PDBFile.writeFile(pdb.topology, state.getPositions(), f)

    summary = {
        "initial_energy_kJ_mol": init_energy,
        "final_energy_kJ_mol": trace[-1]["potential_energy_kJ_mol"],
        "energy_change_kJ_mol": trace[-1]["potential_energy_kJ_mol"] - init_energy,
        "final_ligand_rmsd_A": trace[-1]["ligand_rmsd_A"],
        "final_max_force_kJ_mol_nm": trace[-1]["max_force_kJ_mol_nm"],
        "restraint_k": restraint_k,
        "relaxed_pdb_path": relaxed_pdb
    }
    with open(os.path.join(outdir, "relaxation_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    ax1.plot(df["iteration"], df["potential_energy_kJ_mol"], 'b-', label="Energy (kJ/mol)")
    ax1.set_xlabel("Minimization Iterations")
    ax1.set_ylabel("Potential Energy (kJ/mol)", color='b')
    ax2 = ax1.twinx()
    ax2.plot(df["iteration"], df["ligand_rmsd_A"], 'r--', label="Ligand RMSD (Å)")
    ax2.set_ylabel("Ligand RMSD (Å)", color='r')
    plt.title(f"Relaxation Diagnostics ({os.path.basename(outdir)})")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "relaxation_diagnostics.png"), dpi=200)
    plt.close()

    log(f"[SUCCESS] Final RMSD: {summary['final_ligand_rmsd_A']:.4f} A | Delta E: {summary['energy_change_kJ_mol']:.2f} kJ/mol")
    log_file.close()

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--complex-pdb", required=True)
    parser.add_argument("--system-xml", required=True)
    parser.add_argument("--ligand-indices", required=True)
    parser.add_argument("--protein-indices", required=True)
    parser.add_argument("--outdir", required=True)
    parser.add_argument("--max-iterations", type=int, default=500)
    parser.add_argument("--record-every", type=int, default=10)
    parser.add_argument("--restraint-k", type=float, default=1000.0)
    args = parser.parse_args()

    run_relaxation(args.complex_pdb, args.system_xml, args.ligand_indices, args.protein_indices,
                   args.outdir, args.max_iterations, args.record_every, args.restraint_k)

Overwriting scripts/relax_complex.py


In [ ]:
import os
import urllib.request
import numpy as np
import openmm.app as app
from pdbfixer import PDBFixer

TARGET_NAME = "PARP1"
TARGET_PDB_INPUT = "7ONS"
MANUAL_CENTER = None
BOX_SIZE = [20.0, 20.0, 20.0]

raw_pdb_path = f"data/targets/{TARGET_NAME}_raw.pdb"
clean_pdb_path = f"data/targets/{TARGET_NAME}_clean.pdb"
receptor_pdbqt = f"data/receptors/{TARGET_NAME}_P1.pdbqt"

# 1. Download or locate target PDB
if len(TARGET_PDB_INPUT) == 4 and not os.path.exists(TARGET_PDB_INPUT):
    print(f"⬇️ Downloading PDB structure '{TARGET_PDB_INPUT}' from RCSB...")
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{TARGET_PDB_INPUT}.pdb", raw_pdb_path)
else:
    raw_pdb_path = TARGET_PDB_INPUT

# 2. Extract bound ligand coordinates to locate the binding pocket
ligand_coords = []
with open(raw_pdb_path, "r") as f:
    for line in f:
        if line.startswith("HETATM") and "HOH" not in line:
            try:
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                ligand_coords.append([x, y, z])
            except ValueError:
                pass

if MANUAL_CENTER is not None:
    POCKET_CENTER = MANUAL_CENTER
elif len(ligand_coords) > 0:
    POCKET_CENTER = np.mean(ligand_coords, axis=0).round(2).tolist()
    print(f"🎯 Automatically Detected Pocket Center: X={POCKET_CENTER[0]}, Y={POCKET_CENTER[1]}, Z={POCKET_CENTER[2]}")
else:
    raise ValueError("No bound ligand found! Please set MANUAL_CENTER = [X, Y, Z].")

# 3. Clean protein and add all hydrogens with PDBFixer
print("🔧 Preparing target protein with PDBFixer (adding hydrogens)...")
fixer = PDBFixer(filename=raw_pdb_path)
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)

with open(clean_pdb_path, "w") as f:
    app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

# 4. Generate clean receptor PDBQT with verified AutoDock atom types
receptor_lines = []
with open(clean_pdb_path, "r") as f:
    for line in f:
        if line.startswith("ATOM"):
            elem = line[76:78].strip().upper()
            if not elem:
                name_clean = line[12:16].strip().lstrip("0123456789")
                elem = name_clean[0].upper() if name_clean else "C"

            # Map elements to valid AutoDock 4 types
            if elem == "H": ad_type = "HD"
            elif elem == "C": ad_type = "C"
            elif elem == "N": ad_type = "NA" if "HIS" in line else "N"
            elif elem == "O": ad_type = "OA"
            elif elem == "S": ad_type = "SA"
            elif elem == "P": ad_type = "P"
            elif elem in ["ZN", "FE", "MG", "CA", "MN"]: ad_type = elem.capitalize()
            else: ad_type = elem

            base = line[:66].ljust(70)
            receptor_lines.append(f"{base} 0.000 {ad_type:>2}\n")

with open(receptor_pdbqt, "w") as f:
    f.writelines(receptor_lines)

print(f"✅ Clean Protein PDB: {clean_pdb_path}")
print(f"✅ Receptor PDBQT:   {receptor_pdbqt}")

⬇️ Downloading PDB structure '7ONS' from RCSB...
🎯 Automatically Detected Pocket Center: X=19.32, Y=38.71, Z=21.09
🔧 Preparing target protein with PDBFixer (adding hydrogens)...
✅ Clean Protein PDB: data/targets/PARP1_clean.pdb
✅ Receptor PDBQT:   data/receptors/PARP1_P1.pdbqt


In [ ]:
import glob
import os
import shutil
import zipfile

# 1. Unzip any ZIP files
zip_files = glob.glob("data/ligands/*.zip")
for zf in zip_files:
    print(f"📦 Unzipping: {zf}...")
    with zipfile.ZipFile(zf, 'r') as zip_ref:
        zip_ref.extractall("data/ligands/")

# 2. Flatten nested folders
for root, dirs, files in os.walk("data/ligands"):
    if os.path.abspath(root) != os.path.abspath("data/ligands"):
        for f in files:
            if f.endswith(".sdf") or f.endswith(".npz"):
                src = os.path.join(root, f)
                dst = os.path.join("data/ligands", f)
                if not os.path.exists(dst):
                    shutil.move(src, dst)

# 3. Discover unique candidates by basename (avoids duplicates)
all_found = sorted(glob.glob("data/ligands/*.sdf") + glob.glob("data/ligands/*.npz"))
seen = set()
candidate_files = []
for p in all_found:
    base = os.path.basename(p)
    if base not in seen:
        seen.add(base)
        candidate_files.append(p)

print(f"\n🎉 Total Unique Candidates Found: {len(candidate_files)}")
print("Sample molecules:", [os.path.basename(f) for f in candidate_files[:5]])

📦 Unzipping: data/ligands/individual_sdf-20261115T125641Z-1-001.zip...

🎉 Total Unique Candidates Found: 50
Sample molecules: ['cd_000028_d73121d2ac.sdf', 'cd_000047_bb66809e6c.sdf', 'cd_000113_f8df2831b7.sdf', 'cd_000127_4f4c0f2275.sdf', 'cd_000234_22e8709985.sdf']


In [ ]:
import os
import json
import subprocess
import pandas as pd

batch_records = []

for idx, lig_path in enumerate(candidate_files):
    cand_id = os.path.splitext(os.path.basename(lig_path))[0]
    pair_id = f"{cand_id}__P1"
    pair_outdir = f"outputs/{pair_id}"
    dock_dir = f"{pair_outdir}/docking"
    os.makedirs(dock_dir, exist_ok=True)

    print(f"\n[{idx+1}/{len(candidate_files)}] Processing: {pair_id}...")

    # Step 4A: Local Pocket Docking
    dock_cmd = [
        "python", "scripts/local_dock.py",
        "--receptor-pdbqt", receptor_pdbqt,
        "--ligand-path", lig_path,
        "--center", str(POCKET_CENTER[0]), str(POCKET_CENTER[1]), str(POCKET_CENTER[2]),
        "--size", str(BOX_SIZE[0]), str(BOX_SIZE[1]), str(BOX_SIZE[2]),
        "--outdir", dock_dir,
        "--exhaustiveness", "8",
        "--num-modes", "3"
    ]
    p_dock = subprocess.run(dock_cmd, capture_output=True, text=True)
    if p_dock.returncode != 0 or not os.path.exists(f"{dock_dir}/docked_pose.sdf"):
        print(f"  ❌ Docking failed for {cand_id}: {p_dock.stderr.strip()[:200]}")
        batch_records.append({
            "candidate_id": cand_id, "conformer_id": "P1", "status": "FAILED_DOCKING",
            "error": p_dock.stderr.strip()[:300]
        })
        continue

    # Step 4B: Build Parameterized Complex & OpenMM System
    complex_pdb = f"{pair_outdir}/initial_complex.pdb"
    system_xml = f"{pair_outdir}/system.xml"
    lig_idx = f"{pair_outdir}/ligand_indices.json"
    prot_idx = f"{pair_outdir}/protein_indices.json"

    build_cmd = [
        "python", "scripts/build_complex_system.py",
        "--target-clean-pdb", clean_pdb_path,
        "--docked-sdf", f"{dock_dir}/docked_pose.sdf",
        "--outdir", pair_outdir
    ]
    p_build = subprocess.run(build_cmd, capture_output=True, text=True)
    if p_build.returncode != 0:
        print(f"  ❌ System assembly failed for {cand_id}: {p_build.stderr.strip()[:200]}")
        batch_records.append({
            "candidate_id": cand_id, "conformer_id": "P1", "status": "FAILED_ASSEMBLY",
            "error": p_build.stderr.strip()[:300]
        })
        continue

    # Step 4C: OpenMM Constrained Relaxation & RMSD QC
    relax_cmd = [
        "python", "scripts/relax_complex.py",
        "--complex-pdb", complex_pdb,
        "--system-xml", system_xml,
        "--ligand-indices", lig_idx,
        "--protein-indices", prot_idx,
        "--outdir", pair_outdir,
        "--max-iterations", "500",
        "--record-every", "10",
        "--restraint-k", "1000"
    ]
    p_relax = subprocess.run(relax_cmd, capture_output=True, text=True)
    if p_relax.returncode == 0 and os.path.exists(f"{pair_outdir}/relaxation_summary.json"):
        with open(f"{pair_outdir}/relaxation_summary.json") as f:
            res = json.load(f)
        res["candidate_id"] = cand_id
        res["conformer_id"] = "P1"
        res["status"] = "SUCCESS"
        batch_records.append(res)
        print(f"  ✅ Relaxed: RMSD={res['final_ligand_rmsd_A']:.3f} Å | ΔE={res['energy_change_kJ_mol']:.1f} kJ/mol")
    else:
        print(f"  ❌ OpenMM Relaxation failed for {cand_id}: {p_relax.stderr.strip()[:200]}")
        batch_records.append({
            "candidate_id": cand_id, "conformer_id": "P1", "status": "FAILED_RELAXATION",
            "error": p_relax.stderr.strip()[:300]
        })

# Compile and save batch summary table
summary_df = pd.DataFrame(batch_records)
summary_df.to_csv("outputs/relaxation_summary.csv", index=False)
print(f"\n🎉 Finished batch processing! Master table saved to: outputs/relaxation_summary.csv")


[1/50] Processing: cd_000028_d73121d2ac__P1...
  ❌ OpenMM Relaxation failed for cd_000028_d73121d2ac: /usr/local/lib/python3.13/dist-packages/openmm/app/internal/pdbstructure.py:538: UserWarning: WARNING: duplicate atom (HETATM    2  C   LIG L   1      21.704  34.278  17.690  1.00  0.00           C  ,

[2/50] Processing: cd_000047_bb66809e6c__P1...
  ❌ OpenMM Relaxation failed for cd_000047_bb66809e6c: /usr/local/lib/python3.13/dist-packages/openmm/app/internal/pdbstructure.py:538: UserWarning: WARNING: duplicate atom (HETATM    2  C   LIG L   1      24.784  33.236  15.756  1.00  0.00           C  ,

[3/50] Processing: cd_000113_f8df2831b7__P1...
  ❌ Docking failed for cd_000113_f8df2831b7: /usr/local/lib/python3.13/dist-packages/meeko/preparation.py:765: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWrit

[4/50] Processing: cd_000127_4f4c0f2275__P1...
  ❌ Docking failed for cd_000127_4f4c0f2275: /usr/

KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

# Load batch summary table
df = pd.read_csv("outputs/relaxation_summary.csv")

# Ensure all expected columns exist (fills missing columns with None)
for col in ["final_ligand_rmsd_A", "energy_change_kJ_mol", "final_max_force_kJ_mol_nm", "relaxed_pdb_path"]:
    if col not in df.columns:
        df[col] = None

# Quality Control Filter
def qc_filter(row):
    if row.get("status") != "SUCCESS":
        return f"REJECT_{row.get('status', 'ERROR')}"
    if row.get("energy_change_kJ_mol") is not None and row["energy_change_kJ_mol"] > 0:
        return "FLAG_UNSTABLE_ENERGY"
    if row.get("final_ligand_rmsd_A") is not None and row["final_ligand_rmsd_A"] > 2.5:
        return "FLAG_HIGH_DISPLACEMENT"
    return "PASS_QC"

df["qc_verdict"] = df.apply(qc_filter, axis=1)

print("📋 Stage 4 Batch Quality Control Summary:")
display_cols = [c for c in ["candidate_id", "conformer_id", "status", "final_ligand_rmsd_A", "energy_change_kJ_mol", "qc_verdict"] if c in df.columns]
print(df[display_cols].head(15))

print("\n📊 Status Breakdown:")
print(df["status"].value_counts())

# Export approved candidates to Stage 5 Manifest
passed_df = df[df["qc_verdict"] == "PASS_QC"].copy()
if len(passed_df) > 0:
    stage5_manifest = passed_df[["candidate_id", "conformer_id", "relaxed_pdb_path"]]
    stage5_manifest.to_csv("outputs/stage5_input_manifest.csv", index=False)
    print(f"\n🎯 Total Candidates Evaluated: {len(df)}")
    print(f"✅ Approved for Stage 5 GNINA Scoring: {len(stage5_manifest)}")
    print(f"📄 Saved Stage 5 Manifest to: outputs/stage5_input_manifest.csv")

    first_pass_id = f"{passed_df.iloc[0]['candidate_id']}__{passed_df.iloc[0]['conformer_id']}"
    plot_path = f"outputs/{first_pass_id}/relaxation_diagnostics.png"
    if os.path.exists(plot_path):
        print(f"\n🖼️ Displaying sample diagnostic plot for: {first_pass_id}")
        display(Image(filename=plot_path))
else:
    print("\n⚠️ No candidates passed QC yet. Check the status breakdown above.")

📋 Stage 4 Batch Quality Control Summary:
            candidate_id conformer_id          status final_ligand_rmsd_A  \
0   cd_000028_d73121d2ac           P1  FAILED_DOCKING                None   
1   cd_000047_bb66809e6c           P1  FAILED_DOCKING                None   
2   cd_000113_f8df2831b7           P1  FAILED_DOCKING                None   
3   cd_000127_4f4c0f2275           P1  FAILED_DOCKING                None   
4   cd_000234_22e8709985           P1  FAILED_DOCKING                None   
5   cd_000292_d7d77e778a           P1  FAILED_DOCKING                None   
6   cd_000310_746c5093ed           P1  FAILED_DOCKING                None   
7   cd_000570_ed7563df48           P1  FAILED_DOCKING                None   
8   cd_000614_08e100d67b           P1  FAILED_DOCKING                None   
9   cd_000655_d1e3d203d5           P1  FAILED_DOCKING                None   
10  cd_000682_6e87e83ba8           P1  FAILED_DOCKING                None   
11  cd_000741_34b1040100           